In [0]:
from pyspark.sql import SparkSession
from sedona.spark import SedonaContext
from sedona.spark.utils import KryoSerializer, SedonaKryoRegistrator
from sedona.spark import SedonaContext 

spark = (
    SparkSession.builder
    .appName("roof-top-metrics")
    .config("spark.serializer", KryoSerializer.getName)
    .config("spark.kryo.registrator", SedonaKryoRegistrator.getName)
    # .config("spark.executor.memory", "64g") 
    # .config("spark.executor.cores", "8g") 
    # .config("spark.executor.instances", "4") 
    .config("spark.executor.heartbeatInterval", "60s") 
    .config("spark.network.timeout", "120s") 
    .getOrCreate()
)
spark = SedonaContext.create(spark)

### Load Search Logs

In [0]:
from pyspark.sql.functions import col, expr

search_logs = spark.read.csv("/mnt/opas/opas-source/apt-roof-top-accuracy-improvement/rooftop_prod_logs_run_id_20251209_00032117.csv", header=True, inferSchema=True, sep=",", quote='"', escape='"')

print(f"Total Search Count: {search_logs.count()}")
usa_search_logs = search_logs.filter("country == 'USA'")

usa_total_count = usa_search_logs.count()
usa_rooftop_count = usa_search_logs.filter("rooftop_match == 1").count()
usa_outside_rooftop_count = usa_search_logs.filter("rooftop_match ==0").count()

# Calculate percentages
rooftop_percentage = (usa_rooftop_count / usa_total_count) * 100
outside_rooftop_percentage = (usa_outside_rooftop_count / usa_total_count) * 100

# Print the results
print(f"USA Search Count: {usa_total_count} ({100:.2f}%)")
print(f"USA Search On RoofTop Count: {usa_rooftop_count} ({rooftop_percentage:.2f}%)")
print(f"USA Search Outside RoofTop Count: {usa_outside_rooftop_count} ({outside_rooftop_percentage:.2f}%)")

usa_search_logs.display()

### Country  Specific logs

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Assuming you have a Spark DataFrame named search_logs
# Group by country and rooftop_match to get the count
grouped_result = search_logs.groupBy("country", "rooftop_match").count()

# Calculate total count per country
total_count_per_country = grouped_result.groupBy("country").agg(F.sum("count").alias("total_count"))

# Join the two DataFrames to calculate the percentage
result_with_percentage = grouped_result.join(total_count_per_country, "country") \
    .withColumn("percentage", F.format_number((F.col("count") / F.col("total_count") * 100), 2)) \
    .orderBy(F.col("country").desc())

# Show the result
display(result_with_percentage)


### Load Relocated CSV

In [0]:
relocated_apt_ds = spark.read.csv("/mnt/opas/opas-source/apt-roof-top-accuracy-improvement/relocated_usa_apt_update_1.csv", header=False, inferSchema=True, sep="|", quote='"', escape='"').withColumnRenamed("_c0", "singed_id") \
.withColumnRenamed("_c1", "location") \
.withColumnRenamed("_c2", "relocated_location") \
.withColumnRenamed("_c3", "relocated_bfp") \
.withColumnRenamed("_c4", "uuid") \
.withColumnRenamed("_c5", "comment") \
.withColumnRenamed("_c6", "hn_number") \
.withColumnRenamed("_c7", "street_name") \
.withColumnRenamed("_c8", "apt_id") 

display(relocated_apt_ds)

### Find Relocated APT geometry are exist on given search logs provider rooftop_shape

In [0]:
usa_search_logs_with_relocated_apt = usa_search_logs.join(relocated_apt_ds, col("provider_uid") == col("apt_id"), "leftouter").select(
    usa_search_logs["*"],  # Select all columns from usa_search_logs
    relocated_apt_ds["relocated_location"]  # Select the relocated_geom column
)
display(usa_search_logs_with_relocated_apt)

In [0]:
from pyspark.sql import functions as F

usa_search_metrics_1 = (
    usa_search_logs_with_relocated_apt
    # Normalize empty strings to nulls for relocated_location
    .withColumn(
        "relocated_location_norm",
        F.when(
            (F.col("relocated_location").isNull()) |
            (F.trim("relocated_location") == ""),
            F.lit(None)
        ).otherwise(F.col("relocated_location"))
    )
    # Polygon geometry from rooftop_shape WKT
    .withColumn("rooftop_geom", F.expr("ST_GeomFromWKT(rooftop_shape)"))
    # Point geometry from relocated_location WKT (if present)
    .withColumn(
        "relocated_geom",
        F.when(
            F.col("relocated_location_norm").isNotNull(),
            F.expr("ST_GeomFromWKT(relocated_location_norm)")
        )
    )
    # Fallback point: if relocated_geom is null, build from provider_lon/lat
    .withColumn(
        "effective_point_geom",
        F.when(
            F.col("relocated_geom").isNotNull(),
            F.col("relocated_geom")
        ).when(
            (F.col("provider_lon").isNotNull()) & (F.col("provider_lat").isNotNull()),
            F.expr("ST_Point(provider_lon, provider_lat)")  # X = lon, Y = lat
        )
    )
    # Compute intersects: false if we still don't have a point
    .withColumn(
        "is_within_rooftop_shape",
        F.when(
            F.col("effective_point_geom").isNotNull(),
            F.expr("ST_Intersects(rooftop_geom, effective_point_geom)")
        ).otherwise(F.lit(False))
    )
)

usa_search_metrics_1 = usa_search_metrics_1.drop(
    "rooftop_geom",
    "relocated_geom",
    # "effective_point_geom",
    "relocated_location_norm"
)

display(usa_search_metrics_1)


### Country Specific logs after Relocation is on BFP check

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Assuming you have a Spark DataFrame named search_logs
# Group by country and is_within_rooftop_shape to get the count
metrics_grouped_result = usa_search_metrics_1.groupBy("country", "is_within_rooftop_shape").count()

# Calculate total count per country
total_count_per_country = metrics_grouped_result.groupBy("country").agg(F.sum("count").alias("total_count"))

# Join the two DataFrames to calculate the percentage
result_with_percentage = metrics_grouped_result.join(total_count_per_country, "country") \
    .withColumn("percentage", F.format_number((F.col("count") / F.col("total_count") * 100), 2)) \
    .orderBy(F.col("country").desc())

# Show the result
display(result_with_percentage)